In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install yfinance

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 27.3 MB/s  0:00:00
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 4.1/4.1 MB 65.9 MB/s  0:00:00

   ---------------------------------------- 0/8 [pytz]
   ---------- ----------------------------- 2/8 [websockets]
   ---------- ----------------------------- 2/8 [websockets]
   ---------- ----------------------------- 2/8 [websockets]
   --------------- ------------------------ 3/8 [protobuf]
   --------------- ------------------------ 3/8 [protobuf]
   --------------- ------------------------ 3/8 [protobuf]
   --------------- ------------------------ 3/8 [protobuf]
   -------------------- ------------------- 4/8 [peewee]
   -------------------- ------------------- 4/8 [peewee]
   ------------------------- -------------- 5/8 [lxml]
   ------------------------- -------------- 5/8 [lxml]
   ------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [23]:
import pandas as pd
import numpy as np
import yfinance as yf
import time

# -----------------------------
# SCREENING PARAMETERS
# -----------------------------

# CAPM / WACC assumptions
RISK_FREE_RATE = 0.040
EQUITY_RISK_PREMIUM = 0.050
CORPORATE_TAX_RATE = 0.21

# Market capitalization screen
MIN_MARKET_CAP = 1.0e9
MAX_MARKET_CAP = 20.0e9

# Revenue growth requirement
MIN_REVENUE_CAGR = 0.10

# IMPORTANT:
# Yahoo Finance currently returns a maximum of
# 4 annual financial-statement observations.
# Therefore, use a true 3-year CAGR.
REVENUE_CAGR_YEARS = 3

# ROIC/WACC screen
MIN_ROIC_WACC >= 1.6

# Minimum invested capital
MIN_INVESTED_CAPITAL = 50e6

# Small pause between companies to reduce stress on Yahoo Finance
REQUEST_DELAY = 0.20

# Excluded sectors
EXCLUDED_SECTORS = {
    "Financial Services",
    "Financials",
    "Real Estate"
}

print("Imports and screening settings loaded.")

Imports and screening settings loaded.


In [24]:
SP400_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies"


def get_sp400_tickers():
    """
    Downloads the current S&P MidCap 400 constituent list
    from Wikipedia.

    Uses browser-style headers to avoid HTTP 403 errors.
    """

    try:

        # Browser-style header
        headers = {
            "User-Agent": (
                "Mozilla/5.0 "
                "(Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/120.0 Safari/537.36"
            )
        }

        # Read all tables from the page
        tables = pd.read_html(
            SP400_URL,
            storage_options=headers
        )

        constituent_table = None

        # Find the table containing the S&P 400 members
        for table in tables:

            # Clean column names
            table.columns = [
                str(col).strip()
                for col in table.columns
            ]

            # The constituent table should contain
            # a Symbol column and roughly 400 rows
            if (
                "Symbol" in table.columns
                and len(table) >= 350
            ):

                constituent_table = table.copy()
                break

        if constituent_table is None:

            raise ValueError(
                "Could not find the S&P 400 constituent table."
            )

        # Extract ticker symbols
        tickers = (
            constituent_table["Symbol"]
            .astype(str)
            .str.strip()

            # Yahoo Finance uses "-" rather than "."
            # for tickers such as BRK.B -> BRK-B
            .str.replace(
                ".",
                "-",
                regex=False
            )

            .drop_duplicates()
            .tolist()
        )

        print(
            f"Successfully loaded "
            f"{len(tickers)} S&P 400 tickers."
        )

        return tickers


    except Exception as e:

        print(
            "Error retrieving S&P 400 ticker list:"
        )

        print(e)

        return []


# -----------------------------------------
# CREATE WATCHLIST
# -----------------------------------------

WATCHLIST = get_sp400_tickers()


# -----------------------------------------
# CHECK RESULTS
# -----------------------------------------

print()
print(
    f"Number of stocks in WATCHLIST: "
    f"{len(WATCHLIST)}"
)

print()
print("First 20 tickers:")

print(
    WATCHLIST[:20]
)

Successfully loaded 400 S&P 400 tickers.

Number of stocks in WATCHLIST: 400

First 20 tickers:
['AA', 'AAL', 'AAON', 'ACI', 'ACM', 'ADC', 'AEIS', 'AFG', 'AGCO', 'AHR', 'AIT', 'ALGM', 'ALK', 'ALLY', 'ALSN', 'ALV', 'AM', 'AMG', 'AMH', 'AMKR']


In [25]:
def safe_number(value):
    """
    Converts a value to a finite float.
    Returns None if conversion is not possible.
    """

    try:
        value = float(value)

        if np.isfinite(value):
            return value

        return None

    except (TypeError, ValueError):
        return None


def latest_statement_value(statement, possible_names):
    """
    Finds the most recent available value for one of several
    possible financial statement line-item names.
    """

    if statement is None or statement.empty:
        return None

    for name in possible_names:

        if name in statement.index:

            row = statement.loc[name]

            # Handle duplicate rows if they occur
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]

            row = pd.to_numeric(
                row,
                errors="coerce"
            ).dropna()

            if row.empty:
                continue

            # Yahoo normally supplies dates as columns.
            try:
                row = row.sort_index(ascending=False)
            except Exception:
                pass

            value = safe_number(row.iloc[0])

            if value is not None:
                return value

    return None


def normalize_beta(raw_beta):
    """
    Shrinks observed beta toward 1.0.

    Adjusted Beta =
    67% observed beta + 33% market beta

    Beta is capped between 0.50 and 2.00.
    """

    beta = safe_number(raw_beta)

    if beta is None or beta <= 0:
        return 1.0

    adjusted_beta = (
        0.67 * beta
        + 0.33 * 1.0
    )

    return float(
        np.clip(
            adjusted_beta,
            0.50,
            2.00
        )
    )


def calculate_wacc(
    info,
    balance_sheet,
    income_statement,
    market_cap
):
    """
    Calculates Weighted Average Cost of Capital.
    """

    # -------------------------
    # Cost of equity
    # -------------------------

    beta = normalize_beta(
        info.get("beta")
    )

    cost_of_equity = (
        RISK_FREE_RATE
        + beta * EQUITY_RISK_PREMIUM
    )

    # -------------------------
    # Debt
    # -------------------------

    total_debt = safe_number(
        info.get("totalDebt")
    )

    if total_debt is None or total_debt <= 0:

        total_debt = latest_statement_value(
            balance_sheet,
            [
                "Total Debt",
                "Long Term Debt And Capital Lease Obligation",
                "Long Term Debt"
            ]
        )

    if total_debt is None:
        total_debt = 0.0

    # -------------------------
    # Interest expense
    # -------------------------

    interest_expense = latest_statement_value(
        income_statement,
        [
            "Interest Expense",
            "Interest Expense Non Operating"
        ]
    )

    if interest_expense is not None:
        interest_expense = abs(interest_expense)
    else:
        interest_expense = 0.0

    # -------------------------
    # Cost of debt
    # -------------------------

    if total_debt > 0 and interest_expense > 0:

        cost_of_debt = (
            interest_expense
            / total_debt
        )

        # Guard against unusual accounting values
        cost_of_debt = float(
            np.clip(
                cost_of_debt,
                RISK_FREE_RATE,
                0.12
            )
        )

    else:

        # Default assumed spread of 2%
        cost_of_debt = (
            RISK_FREE_RATE
            + 0.02
        )

    cost_of_debt_after_tax = (
        cost_of_debt
        * (1 - CORPORATE_TAX_RATE)
    )

    # -------------------------
    # Capital weights
    # -------------------------

    capital_base = (
        market_cap
        + total_debt
    )

    if capital_base <= 0:
        return None

    weight_equity = (
        market_cap
        / capital_base
    )

    weight_debt = (
        total_debt
        / capital_base
    )

    wacc = (
        weight_equity * cost_of_equity
        + weight_debt * cost_of_debt_after_tax
    )

    # Conservative minimum WACC
    return max(wacc, 0.055)


def calculate_roic(
    income_statement,
    balance_sheet
):
    """
    ROIC = NOPAT / Invested Capital

    NOPAT = EBIT × (1 - tax rate)

    Invested Capital =
    Total Assets
    - Current Liabilities
    - Cash
    """

    if (
        income_statement is None
        or balance_sheet is None
        or income_statement.empty
        or balance_sheet.empty
    ):
        return None

    # -------------------------
    # EBIT
    # -------------------------

    ebit = latest_statement_value(
        income_statement,
        [
            "Operating Income",
            "EBIT"
        ]
    )

    if ebit is None or ebit <= 0:
        return None

    nopat = (
        ebit
        * (1 - CORPORATE_TAX_RATE)
    )

    # -------------------------
    # Total assets
    # -------------------------

    total_assets = latest_statement_value(
        balance_sheet,
        [
            "Total Assets"
        ]
    )

    if total_assets is None:
        return None

    # -------------------------
    # Current liabilities
    # -------------------------

    current_liabilities = latest_statement_value(
        balance_sheet,
        [
            "Current Liabilities",
            "Total Current Liabilities"
        ]
    )

    if current_liabilities is None:
        current_liabilities = 0.0

    # -------------------------
    # Cash
    # -------------------------

    cash = latest_statement_value(
        balance_sheet,
        [
            "Cash And Cash Equivalents",
            "Cash Cash Equivalents And Short Term Investments",
            "Cash Financial"
        ]
    )

    if cash is None:
        cash = 0.0

    # -------------------------
    # Invested capital
    # -------------------------

    invested_capital = (
        total_assets
        - current_liabilities
        - cash
    )

    if invested_capital < MIN_INVESTED_CAPITAL:
        return None

    roic = (
        nopat
        / invested_capital
    )

    return roic


def check_revenue_cagr(
    income_statement,
    years=3
):
    """
    Calculates revenue CAGR.

    Pass requirement:
    CAGR > MIN_REVENUE_CAGR

    A company is not automatically rejected because
    one individual year had lower revenue.
    """

    if (
        income_statement is None
        or income_statement.empty
    ):
        return None, False, 0

    if "Total Revenue" not in income_statement.index:
        return None, False, 0

    revenue = income_statement.loc[
        "Total Revenue"
    ]

    if isinstance(revenue, pd.DataFrame):
        revenue = revenue.iloc[0]

    revenue = pd.to_numeric(
        revenue,
        errors="coerce"
    ).dropna()

    number_of_observations = len(revenue)

    observations_needed = years + 1

    if number_of_observations < observations_needed:

        return (
            None,
            False,
            number_of_observations
        )

    # Convert column dates
    revenue.index = pd.to_datetime(
        revenue.index
    )

    # Oldest to newest
    revenue = revenue.sort_index()

    # Use most recent 4 observations
    revenue = revenue.iloc[
        -observations_needed:
    ]

    beginning_revenue = float(
        revenue.iloc[0]
    )

    ending_revenue = float(
        revenue.iloc[-1]
    )

    if (
        beginning_revenue <= 0
        or ending_revenue <= 0
    ):

        return (
            None,
            False,
            number_of_observations
        )

    # CAGR
    cagr = (
        ending_revenue
        / beginning_revenue
    ) ** (1 / years) - 1

    # Main growth requirement
    growth_pass = (
        cagr > MIN_REVENUE_CAGR
    )

    return (
        cagr,
        growth_pass,
        number_of_observations
    )


print("Financial calculation functions loaded.")

Financial calculation functions loaded.


In [32]:
def run_screener(
    tickers,
    output_prefix="sp400"
):

    passed = []
    rejected = []

    total = len(tickers)

    for number, sym in enumerate(
        tickers,
        start=1
    ):

        print(
            f"[{number}/{total}] {sym}: ",
            end=""
        )

        company = sym
        sector = ""

        try:

            # -------------------------
            # Create Yahoo ticker
            # -------------------------

            ticker = yf.Ticker(sym)

            # -------------------------
            # Company information
            # -------------------------

            info = ticker.get_info()

            if not info:
                raise ValueError(
                    "Yahoo returned no company information."
                )

            company = info.get(
                "shortName",
                sym
            )

            sector = info.get(
                "sector",
                ""
            )

            country = info.get(
                "country",
                ""
            )

            # -------------------------
            # FILTER 1
            # Sector
            # -------------------------

            if sector in EXCLUDED_SECTORS:

                reason = (
                    f"Excluded sector: {sector}"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(reason)

                continue

            # -------------------------
            # FILTER 2
            # Market capitalization
            # -------------------------

            market_cap = safe_number(
                info.get("marketCap")
            )

            if market_cap is None:

                reason = (
                    "Missing market capitalization"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(reason)

                continue

            if not (
                MIN_MARKET_CAP
                <= market_cap
                <= MAX_MARKET_CAP
            ):

                reason = (
                    "Market cap outside "
                    "$1B-$20B range"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(
                    f"{reason} "
                    f"(${market_cap / 1e9:.2f}B)"
                )

                continue

            # -------------------------
            # FILTER 3
            # United States
            # -------------------------

            if country != "United States":

                reason = (
                    f"Country = {country}"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(reason)

                continue

            # -------------------------
            # Download financials
            # only after basic filters
            # -------------------------

            income_statement = (
                ticker.get_income_stmt(
                    freq="yearly",
                    pretty=True
                )
            )

            balance_sheet = (
                ticker.get_balance_sheet(
                    freq="yearly",
                    pretty=True
                )
            )

            if (
                income_statement.empty
                or balance_sheet.empty
            ):

                reason = (
                    "Financial statements unavailable"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(reason)

                continue

            # -------------------------
            # FILTER 4
            # Revenue CAGR
            # -------------------------

            (
                revenue_cagr,
                growth_pass,
                revenue_observations
            ) = check_revenue_cagr(
                income_statement
            )

            if revenue_cagr is None:

                reason = (
                    f"Insufficient revenue history: "
                    f"{revenue_observations} annual "
                    f"observations; "
                    f"{REVENUE_CAGR_YEARS + 1} required"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(reason)

                continue

            if not growth_pass:

                reason = (
                     f"Revenue CAGR below "
                     f"{MIN_REVENUE_CAGR * 100:.0f}%"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(
                    f"{reason} "
                    f"(CAGR = "
                    f"{revenue_cagr * 100:.2f}%)"
                )

                continue

            # -------------------------
            # WACC
            # -------------------------

            wacc = calculate_wacc(
                info,
                balance_sheet,
                income_statement,
                market_cap
            )

            if wacc is None:

                reason = (
                    "Could not calculate WACC"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(reason)

                continue

            # -------------------------
            # ROIC
            # -------------------------

            roic = calculate_roic(
                income_statement,
                balance_sheet
            )

            if roic is None:

                reason = (
                    "Could not calculate ROIC"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(reason)

                continue

            # -------------------------
            # FILTER 5
            # ROIC / WACC
            # -------------------------

            economic_spread_ratio = (
                roic / wacc
            )

            if economic_spread_ratio < MIN_ROIC_WACC:

                reason = (
                    f"ROIC/WACC below "
                    f"{MIN_ROIC_WACC}"
                )

                rejected.append({
                    "Ticker": sym,
                    "Company": company,
                    "Reason": reason
                })

                print(
                    f"{reason} "
                    f"({economic_spread_ratio:.2f})"
                )

                continue

            # -------------------------
            # COMPANY PASSES
            # -------------------------

            passed.append({

                "Ticker":
                    sym,

                "Company":
                    company,

                "Sector":
                    sector,

                "Market Cap ($B)":
                    round(
                        market_cap / 1e9,
                        2
                    ),

                "Revenue CAGR Years":
                    REVENUE_CAGR_YEARS,

                "Revenue CAGR (%)":
                    round(
                        revenue_cagr * 100,
                        2
                    ),

                "ROIC (%)":
                    round(
                        roic * 100,
                        2
                    ),

                "Adjusted WACC (%)":
                    round(
                        wacc * 100,
                        2
                    ),

                "ROIC/WACC Ratio":
                    round(
                        economic_spread_ratio,
                        2
                    )
            })

            print(
                "PASS "
                f"| CAGR "
                f"{revenue_cagr * 100:.2f}% "
                f"| ROIC "
                f"{roic * 100:.2f}% "
                f"| WACC "
                f"{wacc * 100:.2f}% "
                f"| Ratio "
                f"{economic_spread_ratio:.2f}"
            )

        except Exception as e:

            reason = (
                f"Yahoo/data error: {e}"
            )

            rejected.append({
                "Ticker": sym,
                "Company": company,
                "Reason": reason
            })

            print(reason)

        finally:

            time.sleep(
                REQUEST_DELAY
            )

    # -----------------------------
    # Build final DataFrames
    # -----------------------------

    passed_columns = [
        "Ticker",
        "Company",
        "Sector",
        "Market Cap ($B)",
        "Revenue CAGR Years",
        "Revenue CAGR (%)",
        "ROIC (%)",
        "Adjusted WACC (%)",
        "ROIC/WACC Ratio"
    ]

    rejected_columns = [
        "Ticker",
        "Company",
        "Reason"
    ]

    passed_df = pd.DataFrame(
        passed,
        columns=passed_columns
    )

    rejected_df = pd.DataFrame(
        rejected,
        columns=rejected_columns
    )

    # Sort qualifying companies
    if not passed_df.empty:

        passed_df = (
            passed_df
            .sort_values(
                by="ROIC/WACC Ratio",
                ascending=False
            )
            .reset_index(drop=True)
        )

    # Sort rejected companies
    if not rejected_df.empty:

        rejected_df = (
            rejected_df
            .sort_values(
                by="Ticker"
            )
            .reset_index(drop=True)
        )

    # -----------------------------
    # Save CSV files
    # -----------------------------

    passed_df.to_csv(
        f"{output_prefix}_screen_passed.csv",
        index=False
    )

    rejected_df.to_csv(
        f"{output_prefix}_screen_rejected.csv",
        index=False
    )

    return passed_df, rejected_df


print("Screener function loaded.")

Screener function loaded.


In [33]:
results, rejected = run_screener(
    WATCHLIST,
    output_prefix="sp400"
)

print("\n-----------------------------")
print("SCREEN COMPLETE")
print("-----------------------------")

print(
    f"Stocks screened: {len(WATCHLIST)}"
)

print(
    f"Stocks passing: {len(results)}"
)

print(
    f"Stocks rejected: {len(rejected)}"
)

print("\nQUALIFYING STOCKS:")

display(results)

[1/400] AA: Revenue CAGR below 10% (CAGR = 1.01%)
[2/400] AAL: Revenue CAGR below 10% (CAGR = 3.71%)
[3/400] AAON: ROIC/WACC below 1.6 (0.90)
[4/400] ACI: Revenue CAGR below 10% (CAGR = 2.32%)
[5/400] ACM: Revenue CAGR below 10% (CAGR = 7.07%)
[6/400] ADC: Excluded sector: Real Estate
[7/400] AEIS: Revenue CAGR below 10% (CAGR = -0.85%)
[8/400] AFG: Excluded sector: Financial Services
[9/400] AGCO: Revenue CAGR below 10% (CAGR = -7.29%)
[10/400] AHR: Excluded sector: Real Estate
[11/400] AIT: Revenue CAGR below 10% (CAGR = 4.02%)
[12/400] ALGM: Revenue CAGR below 10% (CAGR = -2.95%)
[13/400] ALK: ROIC/WACC below 1.6 (0.58)
[14/400] ALLY: Excluded sector: Financial Services
[15/400] ALSN: Revenue CAGR below 10% (CAGR = 2.82%)
[16/400] ALV: Country = Sweden
[17/400] AM: Revenue CAGR below 10% (CAGR = 8.32%)
[18/400] AMG: Excluded sector: Financial Services
[19/400] AMH: Excluded sector: Real Estate
[20/400] AMKR: Revenue CAGR below 10% (CAGR = -1.84%)
[21/400] AN: Revenue CAGR below 10% 

,Ticker,Company,Sector,Market Cap ($B),Revenue CAGR Years,Revenue CAGR (%),ROIC (%),Adjusted WACC (%),ROIC/WACC Ratio
0,MANH,"Manhattan Associates, Inc.",Technology,12.00,3,12.13,408.13,8.80,46.36
1,MEDP,"Medpace Holdings, Inc.",Healthcare,17.58,3,20.12,313.79,9.46,33.17
2,IDCC,"InterDigital, Inc.",Technology,8.54,3,22.13,63.56,10.29,6.18
3,QLYS,"Qualys, Inc.",Technology,6.38,3,10.96,46.45,7.69,6.04
4,EXEL,"Exelixis, Inc.",Healthcare,14.45,3,12.93,29.86,7.04,4.24
5,ANF,Abercrombie & Fitch Company,Consumer Cyclical,5.81,3,12.51,32.95,7.85,4.20
6,WING,Wingstop Inc.,Consumer Cyclical,2.85,3,24.91,35.39,9.00,3.93
7,HALO,"Halozyme Therapeutics, Inc.",Healthcare,12.96,3,28.38,28.63,7.82,3.66
8,PCTY,Paylocity Holding Corporation,Technology,8.07,3,14.68,25.58,7.13,3.59
9,CVLT,"Commvault Systems, Inc.",Technology,6.01,3,14.69,25.42,7.71,3.30


In [34]:
rejection_summary = (
    rejected["Reason"]
    .value_counts()
    .reset_index()
)

rejection_summary.columns = [
    "Reason",
    "Number of Stocks"
]

display(rejection_summary)

,Reason,Number of Stocks
0,Revenue CAGR below 10%,191
1,Excluded sector: Financial Services,59
2,ROIC/WACC below 1.6,56
3,Excluded sector: Real Estate,31
4,Market cap outside $1B-$12B range,20
5,Missing market capitalization,5
6,Country = United Kingdom,2
7,Country = Sweden,1
8,Could not calculate ROIC,1
9,Country = Cayman Islands,1
